# ATE for `visit` and `conversion`

Since randomization held (see notebook 01), the difference in average outcome between `treatment=1` and `treatment=0` is a valid estimate of the average treatment effect (ATE) — no adjustment needed.

`visit` and `conversion` are treated as two separate outcomes, each with its own ATE — not a funnel.

In [ ]:
import sys
sys.path.insert(0, "../src")

from uplift.data import load_sample
from uplift.ate import compute_ate

df = load_sample()
df.shape

## Difference-in-means ATE with 95% CI

For a 0/1 outcome, each group's variance is `p(1-p)`. Treatment and control are independent samples, so their variances add:

```
ATE = p_treated - p_control
SE  = sqrt(p_treated*(1-p_treated)/n_treated + p_control*(1-p_control)/n_control)
95% CI = ATE +/- 1.96 * SE
```

In [ ]:
results = {outcome: compute_ate(df, outcome) for outcome in ["visit", "conversion"]}

for outcome, r in results.items():
    print(f"--- {outcome} ---")
    print(f"n_treated={r.n_treated}, n_control={r.n_control}")
    print(f"rate_treated={r.rate_treated:.5f}, rate_control={r.rate_control:.5f}")
    print(f"ATE={r.ate:.5f}  (95% CI: {r.ci_low:.5f} to {r.ci_high:.5f})")
    print(f"relative lift={r.relative_lift:.2%}")
    print()

## Verify: relative CI width, visit vs. conversion

Even though `conversion`'s ATE and absolute CI are numerically smaller, is its CI *proportionally* wider than `visit`'s, confirming that the rarer outcome is noisier relative to its own effect size?

In [ ]:
for outcome, r in results.items():
    rel_half_width = (r.ci_high - r.ate) / r.ate
    print(f"{outcome}: relative half-width = {rel_half_width:.2%}")

## Next: retrospective power / minimum detectable effect (MDE)

Given the sample size and base rates, what's the smallest true effect this experiment could reliably have detected? To be added.